# Notebook 04 — Coverage Gap Analysis
**Project:** WASHLAB Climate-Smart WASH Pilot — Kitui County  
**Analyst:** Davis Mironga  
**Purpose:** Identify communities outside walking distance of any functional borehole.  

**Key inputs:**
- `B_Spatial_Analysis` sheet — 361 GPS-Verified boreholes  
- WorldPop 2020 population raster  
- Kitui ward boundaries (GADM)

**Walking distance threshold:** 2km (confirm with Washlab — some counties use 1km)  
**CRS for distance calculations:** EPSG:32637 (UTM Zone 37N)

---
## Outputs
- Coverage gap map coloured by population in gap
- Ward-level table: total population, covered, in gap, % gap
- GeoJSON of gap polygons for Streamlit app

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
!pip install geopandas shapely rasterio rasterstats folium -q

import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
import rasterstats
from shapely.geometry import Point
from shapely.ops import unary_union
import folium
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/Kitui_WASHLAB/'

UTM_CRS  = 'EPSG:32637'   # UTM Zone 37N — Kenya, for distance calculations
WGS84    = 'EPSG:4326'    # For mapping output
WALK_KM  = 2.0            # Walking distance threshold in km
WALK_M   = WALK_KM * 1000

print(f'Walking threshold: {WALK_KM}km = {WALK_M}m')

In [ ]:
# ── 1. Load borehole data ─────────────────────────────────────────────────────
df = pd.read_excel(
    DRIVE + 'Kitui_Boreholes_Master_Dataset.xlsx',
    sheet_name='B_Spatial_Analysis',
    header=2
)

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df['Longitude'], df['Latitude'])],
    crs=WGS84
)

# Filter to functional boreholes only for coverage calculation
functional = gdf[gdf['Is_Functional'] == True].copy()

print(f'Total GPS-verified boreholes: {len(gdf)}')
print(f'Functional boreholes (for coverage): {len(functional)}')
print(f'Non-functional: {len(gdf) - len(functional)}')

In [ ]:
# ── 2. Project to UTM for accurate distance calculations ──────────────────────
functional_utm = functional.to_crs(UTM_CRS)

# Generate 2km service area buffers
functional_utm['buffer'] = functional_utm.geometry.buffer(WALK_M)
service_areas = gpd.GeoDataFrame(functional_utm, geometry='buffer', crs=UTM_CRS)

# Dissolve all buffers into one coverage polygon
coverage_union = unary_union(service_areas['buffer'])
coverage_gdf = gpd.GeoDataFrame(geometry=[coverage_union], crs=UTM_CRS)

print(f'Coverage area: {coverage_union.area / 1e6:.0f} km²')

In [ ]:
# ── 3. Load ward boundaries ───────────────────────────────────────────────────
# TODO: Update path when boundaries are downloaded
# wards = gpd.read_file(DRIVE + 'boundaries/kitui_wards.shp')

# Placeholder — using sub-county level until ward shapefile is confirmed
# wards = gpd.read_file(DRIVE + 'boundaries/kitui_subcounties.shp')

# IMPORTANT: Verify ward names match borehole dataset before any spatial join
# See docs/data_dictionary.md — Known issue: GADM ward names may differ
print('TODO: Load ward boundaries and verify name alignment')

In [ ]:
# ── 4. Calculate population in gap (once boundaries loaded) ───────────────────
# This cell runs after boundaries and WorldPop raster are available

# gap_polygon = wards.geometry.difference(coverage_union)
# pop_in_gap = rasterstats.zonal_stats(
#     gap_polygon, DRIVE + 'satellite/kitui_worldpop_2020.tif',
#     stats=['sum'], nodata=-9999
# )

print('Population-in-gap calculation ready to run once raster exports complete')

In [ ]:
# ── 5. Quick visual check — boreholes and coverage ────────────────────────────
# Reproject to WGS84 for mapping
coverage_wgs = coverage_gdf.to_crs(WGS84)
functional_wgs = functional.copy()

# Centroid of Kitui County
KITUI_LAT, KITUI_LON = -1.3, 38.0

m = folium.Map(location=[KITUI_LAT, KITUI_LON], zoom_start=8,
               tiles='CartoDB positron')

# Coverage areas
folium.GeoJson(
    coverage_wgs,
    style_function=lambda x: {'fillColor':'#2E75B6','color':'#0B5394',
                               'fillOpacity':0.25,'weight':1},
    name=f'{WALK_KM}km service areas'
).add_to(m)

# Functional boreholes
for _, row in functional_wgs.iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=4, color='#0B5394', fill=True, fill_color='#2E75B6',
        popup=f"{row['Borehole_Name']}<br>{row['Ward']}<br>{row['Management_Type']}"
    ).add_to(m)

folium.LayerControl().add_to(m)
m